# 04 — Corrected SHAP interpretation: Figures 3–4 and cross-split stability

This stage generates SHAP figures and stability summaries using **only** the corrected stored CatBoost models and their registered held-out test sets. This stage does not fit models: its docking-feature inputs are the 44-column models retrained in stage 02 after restoration of `6y1z`/`HTR3A`.

## Two distinct outputs

1. **Figures 3–4:** a preregistered illustrative render from `split_01` (`seed=43`), matching the historical stage-04 split index and using a deterministic 200-molecule SHAP display sample (`seed=42`).
2. **Supplementary stability evidence:** every feature is ranked by mean absolute SHAP on the full held-out test set of every one of the 30 registered structural splits. The audit records top-20 membership counts, not causal effects.

For BBB pass, the held-out set is the BBB-pass subset of each test split. SHAP values explain model predictions; they do not establish protein binding, biological causality, or independent ADMET evidence.

In [ ]:
from __future__ import annotations

import hashlib
from io import BytesIO
import json
import pickle
import platform
from datetime import date
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from catboost import CatBoostRegressor, Pool
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"
FINAL_FIGURES = FIGURES
MODELS = RESULTS / "models"
SPLITS = DATA / "splits"
for directory in (TABLES, FINAL_FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA / "df_final.csv"
ANNOTATION_PATH = DATA / "receptor_panel_annotation.csv"
REGISTRY_PATH = SPLITS / "split_indices.csv"
SPLIT_SUMMARY_PATH = SPLITS / "split_summary.csv"
PROTOCOL_PATH = MODELS / "butina30_protocol.json"
TARGET = "-lgLD50, mol/kg"
LIGAND_ID = "ligand_id"
BBB = "bbb_rule_pass"
PROTEINS = [
    "1m2z", "1pbq", "1xoq", "2rh1", "2vt4", "2ydo", "2z5x", "3b66", "3kk6", "3ln1",
    "3rze", "4djh", "4ey7", "4iar", "4mqs", "4n6h", "5cxv", "5i71", "5tvn", "5u09",
    "5va1", "6cm4", "6kpf", "6kux", "6lqa", "6pdj", "6x3x", "6y1z", "7f8y", "7kwe",
    "7ljd", "7wc9", "7xnk", "7ym8", "8e9y", "8ef6", "8fhs", "8pjk", "8st0", "8wty",
    "8xvk", "8yn3", "9eo4", "V1A",
]
MODEL_SPECS = {
    "Baseline": {"slug": "baseline", "domain": "full_test"},
    "PCA": {"slug": "pca", "domain": "full_test"},
    "ADME": {"slug": "adme", "domain": "full_test"},
    "Plain": {"slug": "plain", "domain": "full_test"},
    "BBB pass": {"slug": "bbb_pass", "domain": "bbb_pass_test"},
}
MODEL_ORDER = list(MODEL_SPECS)
FIG3_MODEL_ORDER = ["Baseline", "Plain", "BBB pass", "ADME"]
FIG3_GROUP_ORDER = ["Fingerprints", "Proteins", "PhysChem", "ADME"]
FIG4_SPECS = {
    "Baseline": ("fig4a.png", "Baseline", (2281, 2818)),
    "BBB pass": ("fig4b.png", "BBB Pass", (2281, 2818)),
    "ADME": ("fig4c.png", "ADME", (2298, 2818)),
}
FIG3_SIZE = (1812, 1412)
FIXED_SPLIT_INDEX = 1
FIXED_SEED = 43
DISPLAY_SAMPLE_SIZE = 200
DISPLAY_SAMPLE_SEED = 42
SHAP_BATCH_SIZE = 128
TOP_K = 20
SHAP_FONT_SCALE = 1.5
SHAP_FEATURE_LABEL_FONT = 13 * SHAP_FONT_SCALE
SHAP_X_LABEL_FONT = 13 * SHAP_FONT_SCALE
SHAP_X_TICK_FONT = 11 * SHAP_FONT_SCALE
SHAP_COLORBAR_LABEL_FONT = 12 * SHAP_FONT_SCALE
SHAP_COLORBAR_TICK_FONT = 11 * SHAP_FONT_SCALE
SHAP_TITLE_FONT = 14 * SHAP_FONT_SCALE


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def escape_latex(value: object) -> str:
    return str(value).replace("\\", r"\textbackslash{}").replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")

for required in (DATA_PATH, ANNOTATION_PATH, REGISTRY_PATH, SPLIT_SUMMARY_PATH, PROTOCOL_PATH):
    assert required.is_file(), required
print("Repository root: current working copy")
print(f"Fig. 3/Fig. 4 display split: split_{FIXED_SPLIT_INDEX:02d}, seed={FIXED_SEED}")
print(f"SHAP stability audit: {len(MODEL_ORDER)} models × 30 held-out test sets")

In [ ]:
df = pd.read_csv(DATA_PATH)
registry = pd.read_csv(REGISTRY_PATH)
split_summary = pd.read_csv(SPLIT_SUMMARY_PATH).set_index("split_index")
annotation = pd.read_csv(ANNOTATION_PATH)

assert len(df) == 12650 and df[LIGAND_ID].nunique() == 12650
assert set(registry["split_index"].astype(int)) == set(range(30))
assert int(split_summary.loc[FIXED_SPLIT_INDEX, "seed"]) == FIXED_SEED
assert set(annotation["panel_col"].astype(str)) == set(PROTEINS)

GENE_MAP = dict(zip(annotation["panel_col"].astype(str), annotation["gene"].astype(str)))
GENE_MAP["9uwl"] = "AVPR1A"
GENE_MAP["V1A"] = "AVPR1A"
df_by_id = df.set_index(LIGAND_ID, drop=False)


def display_feature(feature: str) -> str:
    return GENE_MAP.get(feature, feature)


def feature_group(feature: str) -> str:
    if feature.startswith("FP_"):
        return "Fingerprints"
    if feature in PROTEINS or feature.startswith("PC"):
        return "Proteins"
    if feature in {"MW, g/mol", "MW", "logP"}:
        return "PhysChem"
    return "ADME"


def split_test_frame(split_index: int, model_name: str) -> pd.DataFrame:
    ids = registry.loc[
        registry["split_index"].eq(split_index) & registry["set"].eq("test"), LIGAND_ID
    ].astype(int).tolist()
    frame = df_by_id.loc[ids].reset_index(drop=True).copy()
    if model_name == "BBB pass":
        frame = frame.loc[frame[BBB].eq(1)].reset_index(drop=True)
    return frame


def load_model_inputs(split_index: int, model_name: str) -> tuple[CatBoostRegressor, list[str], pd.DataFrame, Path]:
    spec = MODEL_SPECS[model_name]
    model_dir = MODELS / f"split_{split_index:02d}" / spec["slug"]
    preprocess_path = model_dir / "preprocess.json"
    model_path = model_dir / "model.cbm"
    assert preprocess_path.is_file() and model_path.is_file(), model_dir
    preprocess = json.loads(preprocess_path.read_text(encoding="utf-8"))
    features = list(preprocess["features"])
    frame = split_test_frame(split_index, model_name)
    if preprocess["pca"]:
        with (model_dir / "pca.pkl").open("rb") as handle:
            pca = pickle.load(handle)
        pca_input_features = list(pca.feature_names_in_)
        fill_values = pd.Series(preprocess["protein_fill_values"], dtype=float).reindex(pca_input_features)
        assert fill_values.notna().all(), (split_index, model_name, "Missing PCA fill values")
        components = pca.transform(frame[pca_input_features].fillna(fill_values))
        for component_index in range(components.shape[1]):
            frame[f"PC{component_index + 1}"] = components[:, component_index]
    missing = sorted(set(features) - set(frame.columns))
    assert not missing, (model_name, split_index, missing)
    x = frame[features].copy()
    model = CatBoostRegressor()
    model.load_model(model_path)

    prediction_path = model_dir / f"test_predictions_{spec['domain']}.csv"
    stored = pd.read_csv(prediction_path)
    assert stored[LIGAND_ID].astype(int).tolist() == frame[LIGAND_ID].astype(int).tolist()
    reconstructed = model.predict(x)
    assert np.allclose(reconstructed, stored["y_pred"].to_numpy(float), rtol=0, atol=1e-12), (model_name, split_index)
    return model, features, x, model_dir


def shap_aggregate(model: CatBoostRegressor, x: pd.DataFrame, features: list[str]) -> tuple[np.ndarray, dict[str, float]]:
    abs_sum = np.zeros(len(features), dtype=np.float64)
    group_abs_sum = {group: 0.0 for group in FIG3_GROUP_ORDER}
    group_indices = {
        group: np.asarray([index for index, feature in enumerate(features) if feature_group(feature) == group], dtype=int)
        for group in FIG3_GROUP_ORDER
    }
    for start in range(0, len(x), SHAP_BATCH_SIZE):
        batch = x.iloc[start:start + SHAP_BATCH_SIZE]
        values = model.get_feature_importance(Pool(batch, feature_names=features), type="ShapValues")[:, :-1]
        abs_sum += np.abs(values).sum(axis=0)
        for group, indices in group_indices.items():
            if len(indices):
                group_abs_sum[group] += np.abs(values[:, indices].sum(axis=1)).sum()
    n_rows = len(x)
    return abs_sum / n_rows, {
        group: (group_abs_sum[group] / n_rows if len(group_indices[group]) else np.nan)
        for group in FIG3_GROUP_ORDER
    }

print({"corrected_rows": len(df), "registered_splits": registry["split_index"].nunique(), "annotation_rows": len(annotation)})

## 30-split SHAP stability audit

For every stored model, SHAP values are calculated on all molecules in its held-out test domain. The 200-molecule sample used later for figures is not used here. Feature ranks are based on mean absolute SHAP within each model/split test domain; `top20_count` is the number of registered splits in which the feature ranked 1–20.

In [ ]:
feature_rows = []
group_rows = []
membership_rows = []
model_input_hashes = {}

for split_index in range(30):
    seed = int(split_summary.loc[split_index, "seed"])
    for model_name in MODEL_ORDER:
        model, features, x, model_dir = load_model_inputs(split_index, model_name)
        mean_abs, group_values = shap_aggregate(model, x, features)
        ordering = np.lexsort((np.asarray(features, dtype=str), -mean_abs))
        ranks = np.empty(len(features), dtype=int)
        ranks[ordering] = np.arange(1, len(features) + 1)
        domain = MODEL_SPECS[model_name]["domain"]

        for group in FIG3_GROUP_ORDER:
            group_rows.append({
                "split_index": split_index,
                "seed": seed,
                "model": model_name,
                "evaluation_domain": domain,
                "n_test": len(x),
                "group": group,
                "mean_abs_grouped_shap": group_values[group],
                "n_features": sum(feature_group(feature) == group for feature in features),
            })
        for index, feature in enumerate(features):
            row = {
                "split_index": split_index,
                "seed": seed,
                "model": model_name,
                "evaluation_domain": domain,
                "n_test": len(x),
                "feature": feature,
                "display_feature": display_feature(feature),
                "group": feature_group(feature),
                "mean_abs_shap": float(mean_abs[index]),
                "rank": int(ranks[index]),
                "in_top20": bool(ranks[index] <= TOP_K),
            }
            feature_rows.append(row)
            if ranks[index] <= TOP_K:
                membership_rows.append(row)
        model_input_hashes[f"split_{split_index:02d}/{MODEL_SPECS[model_name]['slug']}/model.cbm"] = sha256(model_dir / "model.cbm")
        model_input_hashes[f"split_{split_index:02d}/{MODEL_SPECS[model_name]['slug']}/preprocess.json"] = sha256(model_dir / "preprocess.json")
        if (model_dir / "pca.pkl").is_file():
            model_input_hashes[f"split_{split_index:02d}/{MODEL_SPECS[model_name]['slug']}/pca.pkl"] = sha256(model_dir / "pca.pkl")
    print(f"SHAP stability: completed split_{split_index:02d} (seed={seed})", flush=True)

feature_by_split = pd.DataFrame(feature_rows)
group_by_split = pd.DataFrame(group_rows)
top20_membership = pd.DataFrame(membership_rows)
assert len(top20_membership) == 30 * len(MODEL_ORDER) * TOP_K
assert feature_by_split.groupby(["split_index", "model"]).size().ge(TOP_K).all()

print({"feature_rows": len(feature_by_split), "group_rows": len(group_by_split), "top20_memberships": len(top20_membership)})


In [ ]:
group_summary = (
    group_by_split.groupby(["model", "evaluation_domain", "group", "n_features"], dropna=False)
    .agg(
        n_splits=("split_index", "nunique"),
        n_test_mean=("n_test", "mean"),
        mean_abs_grouped_shap_mean=("mean_abs_grouped_shap", "mean"),
        mean_abs_grouped_shap_sd=("mean_abs_grouped_shap", "std"),
        mean_abs_grouped_shap_median=("mean_abs_grouped_shap", "median"),
        mean_abs_grouped_shap_min=("mean_abs_grouped_shap", "min"),
        mean_abs_grouped_shap_max=("mean_abs_grouped_shap", "max"),
    )
    .reset_index()
)

feature_stability = (
    feature_by_split.groupby(["model", "evaluation_domain", "feature", "display_feature", "group"], dropna=False)
    .agg(
        n_splits=("split_index", "nunique"),
        top20_count=("in_top20", "sum"),
        mean_abs_shap_mean=("mean_abs_shap", "mean"),
        mean_abs_shap_sd=("mean_abs_shap", "std"),
        median_rank=("rank", "median"),
        mean_rank=("rank", "mean"),
        minimum_rank=("rank", "min"),
        maximum_rank=("rank", "max"),
    )
    .reset_index()
)
feature_stability["top20_frequency_percent"] = 100 * feature_stability["top20_count"] / feature_stability["n_splits"]
conditional_rank = (
    feature_by_split.loc[feature_by_split["in_top20"]]
    .groupby(["model", "evaluation_domain", "feature"], dropna=False)["rank"]
    .median()
    .rename("median_rank_when_top20")
    .reset_index()
)
feature_stability = feature_stability.merge(conditional_rank, on=["model", "evaluation_domain", "feature"], how="left", validate="one_to_one")
feature_stability = feature_stability.sort_values(
    ["model", "top20_count", "mean_abs_shap_mean", "feature"],
    ascending=[True, False, False, True],
    kind="stable",
).reset_index(drop=True)

supplementary_top20 = (
    feature_stability.groupby("model", group_keys=False, sort=False)
    .head(TOP_K)
    .reset_index(drop=True)
)

group_summary.to_csv(TABLES / "shap_group_importance.csv", index=False)
supplementary_top20.to_csv(TABLES / "shap_top20_stability_supplementary.csv", index=False)
assert len(supplementary_top20) == TOP_K * len(MODEL_ORDER)
assert set(supplementary_top20["n_splits"]) == {30}
display(supplementary_top20[["model", "feature", "display_feature", "group", "top20_count", "top20_frequency_percent", "mean_abs_shap_mean", "median_rank"]])


## Supplementary Table S6 source

The reader-facing table lists the 20 features with the highest top-20 occurrence count within each model. Ties are resolved by mean absolute SHAP, then by feature name. The complete table, including features with zero top-20 appearances, remains available as a machine-readable CSV.

In [ ]:
# The compact CSV is the Supplementary evidence source; LaTeX fragments are maintained only with the article source, not in this package.
print("Saved final SHAP stability evidence to results/tables/shap_top20_stability_supplementary.csv")


## Fixed-split Figure 3 and Figure 4 render

The following figures retain the historical visual role only. They use `split_01` / seed 43 because the legacy stage 04 used its historical split 01, not because corrected performance or SHAP results selected this split. The 30-split tables above are the stability evidence.

In [ ]:
def save_exact(fig, path: Path, target_size: tuple[int, int], dpi: int = 300) -> None:
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    with Image.open(buffer) as image:
        rgb = image.convert("RGB")
        if rgb.size != target_size:
            rgb = rgb.resize(target_size, Image.Resampling.LANCZOS)
        rgb.save(path, dpi=(dpi, dpi))

fixed_group_rows = []
fixed_feature_rows = []
fixed_cache = {}
sample_id_rows = []
for model_name in MODEL_ORDER:
    model, features, x_full, model_dir = load_model_inputs(FIXED_SPLIT_INDEX, model_name)
    sample = x_full.sample(min(DISPLAY_SAMPLE_SIZE, len(x_full)), random_state=DISPLAY_SAMPLE_SEED)
    sample_ids = split_test_frame(FIXED_SPLIT_INDEX, model_name).loc[sample.index, LIGAND_ID].astype(int).tolist()
    values_with_base = model.get_feature_importance(Pool(sample, feature_names=features), type="ShapValues")
    values = values_with_base[:, :-1]
    fixed_cache[model_name] = {"x": sample.reset_index(drop=True), "values": values, "features": features}
    for display_order, ligand_id in enumerate(sample_ids, start=1):
        sample_id_rows.append({
            "split_index": FIXED_SPLIT_INDEX,
            "seed": FIXED_SEED,
            "model": model_name,
            "evaluation_domain": MODEL_SPECS[model_name]["domain"],
            "display_sample_seed": DISPLAY_SAMPLE_SEED,
            "display_order": display_order,
            "ligand_id": ligand_id,
        })
    for group in FIG3_GROUP_ORDER:
        indices = [index for index, feature in enumerate(features) if feature_group(feature) == group]
        value = float(np.mean(np.abs(np.sum(values[:, indices], axis=1)))) if indices else np.nan
        fixed_group_rows.append({"model": model_name, "group": group, "mean_abs_grouped_shap": value, "n_features": len(indices)})
    mean_abs = np.mean(np.abs(values), axis=0)
    order = np.lexsort((np.asarray(features, dtype=str), -mean_abs))[:30]
    for rank, index in enumerate(order, start=1):
        fixed_feature_rows.append({
            "model": model_name,
            "rank": rank,
            "feature": features[index],
            "display_feature": display_feature(features[index]),
            "group": feature_group(features[index]),
            "mean_abs_shap": float(mean_abs[index]),
        })

fixed_group_long = pd.DataFrame(fixed_group_rows)
fixed_group_wide = fixed_group_long.pivot(index="group", columns="model", values="mean_abs_grouped_shap").reindex(index=FIG3_GROUP_ORDER, columns=MODEL_ORDER)
assert len(sample_id_rows) == DISPLAY_SAMPLE_SIZE * len(MODEL_ORDER)
print("Prepared fixed split-01 SHAP display data")

In [ ]:
importance = fixed_group_long.pivot(index="group", columns="model", values="mean_abs_grouped_shap").reindex(index=FIG3_GROUP_ORDER, columns=FIG3_MODEL_ORDER)
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    importance,
    annot=importance.round(3),
    fmt=".3f",
    cmap=sns.color_palette("Blues", as_cmap=True),
    mask=importance.isna(),
    cbar=True,
    ax=ax,
    annot_kws={"fontsize": 22},
)
na_y, na_x = np.where(importance.isna().values)
for y, x in zip(na_y, na_x):
    ax.text(x + 0.5, y + 0.5, "N/A", ha="center", va="center", fontsize=22, color="black")
ax.set_xlabel("Models", fontsize=24)
ax.set_ylabel("Feature Groups", fontsize=24)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=22)
ax.set_yticklabels(ax.get_yticklabels(), rotation=90, fontsize=22)
ax.collections[0].colorbar.ax.tick_params(labelsize=18)
fig3_path = FINAL_FIGURES / "fig3.png"
save_exact(fig, fig3_path, FIG3_SIZE, dpi=120)
print(f"Saved {fig3_path.relative_to(ROOT)} size={Image.open(fig3_path).size}")

In [ ]:
figure_paths = {"fig3": fig3_path}
for model_name, (filename, title_name, target_size) in FIG4_SPECS.items():
    cache = fixed_cache[model_name]
    x_display = cache["x"].copy()
    x_display.columns = [display_feature(feature) for feature in x_display.columns]
    plt.figure(figsize=(12, 8))
    shap.summary_plot(cache["values"], x_display, max_display=20, show=False, rng=DISPLAY_SAMPLE_SEED)
    ax = plt.gca()
    ax.tick_params(axis="x", labelsize=SHAP_X_TICK_FONT)
    ax.set_xlabel(ax.get_xlabel(), fontsize=SHAP_X_LABEL_FONT)
    for label in ax.get_yticklabels():
        label.set_fontsize(SHAP_FEATURE_LABEL_FONT)
    for colorbar_ax in plt.gcf().axes:
        if colorbar_ax is not ax:
            colorbar_ax.tick_params(labelsize=SHAP_COLORBAR_TICK_FONT)
            colorbar_ax.set_ylabel(colorbar_ax.get_ylabel(), fontsize=SHAP_COLORBAR_LABEL_FONT)
    ax.set_title(f"SHAP: {title_name} Model (Detailed)", fontsize=SHAP_TITLE_FONT, fontweight="bold")
    plt.tight_layout()
    out_path = FINAL_FIGURES / filename
    save_exact(plt.gcf(), out_path, target_size, dpi=300)
    figure_paths[filename.removesuffix('.png')] = out_path
    print(f"Saved {out_path.relative_to(ROOT)} size={Image.open(out_path).size}")

In [ ]:
final_outputs = [
    TABLES / "shap_group_importance.csv",
    TABLES / "shap_top20_stability_supplementary.csv",
    *figure_paths.values(),
]
for path in final_outputs:
    assert path.is_file() and path.stat().st_size > 0, path
for image_path in figure_paths.values():
    with Image.open(image_path) as image:
        image.verify()
assert len(pd.read_csv(TABLES / "shap_top20_stability_supplementary.csv")) == 100
print("Corrected SHAP Figures 3–4 and final SHAP stability CSV: PASS")


In [ ]:
# Artifact preview — show the completed SHAP table and every manuscript figure.
from IPython.display import Image as NotebookImage, Markdown, display

assert (TABLES / "shap_top20_stability_supplementary.csv").is_file()
display(Markdown("## Supplementary SHAP stability table"))
display(supplementary_top20[[
    "model", "display_feature", "group", "top20_count", "top20_frequency_percent",
    "mean_abs_shap_mean", "median_rank",
]])
for title, path in (("Figure 3", FINAL_FIGURES / "fig3.png"),
                    ("Figure 4a", FINAL_FIGURES / "fig4a.png"),
                    ("Figure 4b", FINAL_FIGURES / "fig4b.png"),
                    ("Figure 4c", FINAL_FIGURES / "fig4c.png")):
    assert path.is_file() and path.stat().st_size > 0, path
    display(Markdown(f"## {title}: `{path.relative_to(ROOT)}`"))
    display(NotebookImage(filename=str(path)))
